In [1]:
install.packages("vegan")

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [5]:
library(vegan)
library(ggplot2)

Loading required package: permute



In [111]:
counts <- read.table("matrix_wgs_rnaseq_species.csv", header = TRUE, sep = ",",
                     row.names = 1, check.names = FALSE)
meta <- read.table("../wgs_rnaseq_biosamples_metagenomes.csv", header = TRUE, sep = ",",
                   check.names = FALSE)

In [113]:
# Exact values to keep (one or several)
values_to_keep <- c("RNA-Seq")

# Filter metadata rows
meta_sub <- meta[meta$ASSAY_TYPE %in% values_to_keep, ]

# Keep matrix columns whose name is in meta$run
keep_runs <- intersect(meta_sub$run, colnames(counts))
counts_sub <- counts[, keep_runs, drop = FALSE]

# Reorder metadata to follow the matrix column order
meta_sub <- meta_sub[match(colnames(counts_sub), meta_sub$run), ]

# Sanity checks
stopifnot(identical(colnames(counts_sub), meta_sub$run))
message(length(setdiff(meta_sub$run, colnames(counts))), " runs absent from the matrix")

0 runs absent from the matrix



In [115]:
counts_t <- t(counts_sub)

summary(rowSums(counts_t))
# counts_t <- counts_t[rowSums(counts_t) >= 1000, ]
# dim(counts_t)   

# # Remove runs without any viral read (division by zero otherwise)
# counts_t <- counts_t[rowSums(counts_t) > 0, ]

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
      1     243    1394   12443    6951  720271 

In [120]:
# counts_t["SRR5256710", ]
counts$SRR5498916

NULL

In [46]:
print(counts_t)

            Ackermannviridae Adenoviridae Adintoviridae Adomaviridae
DRR067250                  0            0             0            0
DRR067251                  0            0             0            0
DRR067252                  0            0             0            0
DRR067253                  0            0             0            0
DRR067254                  0            0             0            0
DRR067255                  0            0             0            0
DRR067256                  0            0             0            0
DRR067257                  0            0             0            0
DRR067259                  0            0             0            0
DRR067260                  0            0             0            0
DRR067261                  0            0             0            0
DRR067262                  0            0             0            0
DRR067264                  0            0             0            0
DRR067265                  0      

# PCoA

In [73]:
# Relative abundance per run
rel <- decostand(counts_t, method = "total")

# Bray-Curtis and PCoA
bc <- vegdist(rel, method = "bray")

pcoa <- cmdscale(bc, k = 2, eig = TRUE)

# Variance explained per axis
var_expl <- pcoa$eig / sum(pcoa$eig[pcoa$eig > 0]) * 100

# Display PCoA based on metadata

In [59]:
# Coordinates as a data frame, with run IDs as a column
scores <- data.frame(run = rownames(pcoa$points),
                    PCoA1 = pcoa$points[, 1],
                    PCoA2 = pcoa$points[, 2])
    
# Check IDs before joining
print(setdiff(scores$run, meta$run))
    
# Join coordinates and metadata
plot_df <- merge(scores, meta, by = "run", all.x = TRUE)
    
# Coloured by the chosen metadata column
color_col <- "date_bin"
highlight_value <- "> 2019"
    
# Flag highlighted runs
plot_df$highlight <- !is.na(plot_df[[color_col]]) &
                    plot_df[[color_col]] == highlight_value
    
p <- ggplot(plot_df, aes(PCoA1, PCoA2)) +
    geom_point(data = plot_df[!plot_df$highlight, ],
            colour = "grey80", size = 1.5, alpha = 0.5) +
    geom_point(data = plot_df[plot_df$highlight, ],
            aes(colour = .data[[color_col]]), size = 2.5, alpha = 0.9) +
    scale_colour_manual(values = setNames("#D55E00", highlight_value)) +
    labs(x = sprintf("PCoA1 (%.1f%%)", var_expl[1]),
        y = sprintf("PCoA2 (%.1f%%)", var_expl[2]),
        colour = color_col,
        title = sprintf("PCoA of viral family composition (Bray-Curtis), highlighted: %s",
                        highlight_value)) +
    theme_bw()
    
ggsave("pcoa_family_date.png", p, width = 7, height = 5, dpi = 300)

character(0)


In [60]:
# Coordinates as a data frame, with run IDs as a column
scores <- data.frame(run = rownames(pcoa$points),
                     PCoA1 = pcoa$points[, 1],
                     PCoA2 = pcoa$points[, 2])

# Check IDs before joining
print(setdiff(scores$run, meta$run))

# Join coordinates and metadata
plot_df <- merge(scores, meta, by = "run", all.x = TRUE)

# Coloured by the chosen metadata column
color_col <- "ASSAY_TYPE"

p <- ggplot(plot_df, aes(PCoA1, PCoA2, colour = .data[[color_col]])) +
  geom_point(size = 2, alpha = 0.8) +
  labs(x = sprintf("PCoA1 (%.1f%%)", var_expl[1]),
       y = sprintf("PCoA2 (%.1f%%)", var_expl[2]),
       colour = color_col,
       title = "PCoA of viral family composition (Bray-Curtis)") +
  theme_bw()

ggsave("pcoa_family_metagenome.png", p) 

character(0)


Saving 6.67 x 6.67 in image


# Evaluate principals drivers

In [64]:
# Axis coordinates (same row order as rel)
ax <- pcoa$points

# Covariance of each family with each axis (families x axes)
cov_ax <- cov(rel, ax)

fam_df <- data.frame(family = colnames(rel),
                     cov1 = cov_ax[, 1],
                     cov2 = cov_ax[, 2],
                     mean_rel = colMeans(rel),
                     prevalence = colSums(rel > 0) / nrow(rel))

# Share of the total absolute covariance on axis 1 (sums to 100%)
fam_df$contrib1 <- abs(fam_df$cov1) / sum(abs(fam_df$cov1)) * 100

# Rank by absolute covariance, sign gives the side of the axis
fam_df <- fam_df[order(abs(fam_df$cov1), decreasing = TRUE), ]
head(fam_df, 30)

print(fam_df$family == "Coronaviridae")

,family,cov1,cov2,mean_rel,prevalence,contrib1
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Kyanoviridae,Kyanoviridae,0.0492217719,1.557734e-02,0.1458068988,0.55201549,26.76256876
Myoviridae,Myoviridae,-0.0375585860,1.409045e-02,0.1007636272,0.28991375,20.42113074
Autographiviridae,Autographiviridae,0.0239512206,1.553618e-02,0.1145201496,0.73737018,13.02261506
Siphoviridae,Siphoviridae,-0.0236489906,4.519356e-03,0.0774325592,0.40239394,12.85828835
Podoviridae,Podoviridae,-0.0221567006,5.356363e-03,0.0754519090,0.41348354,12.04690930
Phycodnaviridae,Phycodnaviridae,0.0041509172,-5.200081e-02,0.2101891367,0.88505545,2.25691200
Zobellviridae,Zobellviridae,0.0039964736,-5.994355e-05,0.0202796701,0.39940151,2.17293882
Straboviridae,Straboviridae,0.0027633715,-1.101686e-03,0.0138556427,0.51064953,1.50248390
Herpesviridae,Herpesviridae,-0.0024235996,-3.798804e-04,0.0185765877,0.57102623,1.31774516


  [1] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
 [13] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
 [25]  TRUE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
 [37] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
 [49] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
 [61] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
 [73] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
 [85] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
 [97] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
[109] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
[121] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
[133] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
[145] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALS

# Multivariate analysis on raw data

In [93]:
clr_mat <- decostand(counts_t, method = "rclr")
pca <- rda(clr_mat)

In [89]:
# Eigenvalues and variance explained per axis
eig <- pca$CA$eig
var_expl <- eig / sum(eig) * 100

# Sample coordinates (distance scaling), run IDs as a column
sc <- scores(pca, display = "sites", choices = 1:2, scaling = 1)
scores_df <- data.frame(run = rownames(sc),
                        PCA1 = sc[, 1],
                        PCA2 = sc[, 2])

# Check IDs before joining
print(setdiff(scores_df$run, meta$run))

# Join coordinates and metadata
plot_df <- merge(scores_df, meta, by = "run", all.x = TRUE)

# Coloured by the chosen metadata column
color_col <- "date_bin"
highlight_value <- "> 2019"
transfo <- "robust CLR"

# Flag highlighted runs
plot_df$highlight <- !is.na(plot_df[[color_col]]) &
                     plot_df[[color_col]] == highlight_value

p <- ggplot(plot_df, aes(PCA1, PCA2)) +
  geom_point(data = plot_df[!plot_df$highlight, ],
             colour = "grey80", size = 1.5, alpha = 0.5) +
  geom_point(data = plot_df[plot_df$highlight, ],
             aes(colour = .data[[color_col]]), size = 2.5, alpha = 0.9) +
  scale_colour_manual(values = setNames("#D55E00", highlight_value)) +
  labs(x = sprintf("PCA1 (%.1f%%)", var_expl[1]),
       y = sprintf("PCA2 (%.1f%%)", var_expl[2]),
       colour = color_col,
       title = sprintf("PCA of viral family composition (%s)",
                       transfo, highlight_value)) +
  theme_bw()

ggsave("../rnaseq_pca_family_date.png", p, width = 7, height = 5, dpi = 300)

character(0)


Warning message in sprintf("PCA of viral family composition (%s)", transfo, highlight_value):
“one argument not used by format 'PCA of viral family composition (%s)'”


In [95]:
# Plot, coloured by the chosen metadata column
color_col <- "organism_tax_name"

p <- ggplot(plot_df, aes(PCA1, PCA2, colour = .data[[color_col]])) +
  geom_point(size = 2, alpha = 0.8) +
  labs(x = sprintf("PCA1 (%.1f%%)", var_expl[1]),
       y = sprintf("PCA2 (%.1f%%)", var_expl[2]),
       colour = color_col,
       title = "PCA of viral family composition (%s)") +
  theme_bw()

ggsave("../rnaseq_pca_family_metagenome.png", p) 

Saving 6.67 x 6.67 in image


In [122]:
# Loadings: eigenvectors, unit norm per axis
v <- pca$CA$v[, 1:2]

contrib1 <- v[, 1]^2 * 100

sort(contrib1, decreasing = TRUE)[1:30]

Caudoviricetes sp.              uncultured marine virus 
                           2.7059366                            2.6914533 
                    Chronagont virus                     uncultured virus 
                           2.2046287                            2.0439117 
                       Ripador virus                uncultured cyanophage 
                           1.8881088                            1.5258209 
                  Picornavirales sp.             Synechococcus cyanophage 
                           1.1664003                            1.1584171 
uncultured Mediterranean phage uvMED                   Picornaviridae sp. 
                           1.0586181                            1.0324014 
                      Ripakrox virus     Myoviridae environmental samples 
                           0.9111212                            0.8210420 
                   phage 023Pt_psg01   Prochlorococcus cyanophage P-SSM10 
                           0.8095203                            0.7292323 
                      Ripojast virus                   Picobirnavirus sp. 
                           0.6929937                            0.6854364 
                       Ripadiz virus                       Ripagrog virus 
                           0.6570436                            0.6539505 
                      Ripathur virus     Megaviridae environmental sample 
                           0.6489954                            0.6487583 
                       Ripabiq virus                       Ripastem virus 
                           0.6442149                            0.6038612 
      uncultured Mediterranean phage                       Ripathuj virus 
                           0.5956803                            0.5923064 
                      Ripothyf virus                       Ripakrev virus 
                           0.5812018                            0.5802032 
                       Ripadip virus                     uncultured phage 
                           0.5698097                            0.5609414 
          Clinch tombus-like virus 1                  Lowelvirus tuscon4d 
                           0.5255302                            0.5183847

# PCA and CA on raw data

In [27]:
pca <- rda(counts_t)

In [ ]:
# Eigenvalues and variance explained per axis
eig <- pca$CA$eig
var_expl <- eig / sum(eig) * 100

# Sample coordinates (distance scaling), run IDs as a column
sc <- scores(pca, display = "sites", choices = 1:2, scaling = 1)
scores_df <- data.frame(run = rownames(sc),
                        PCA1 = sc[, 1],
                        PCA2 = sc[, 2])

# Check IDs before joining
print(setdiff(scores_df$run, meta$run))

# Join coordinates and metadata
plot_df <- merge(scores_df, meta, by = "run", all.x = TRUE)

# Coloured by the chosen metadata column
color_col <- "date_bin"
highlight_value <- "> 2019"
transfo <- "robust CLR"

# Flag highlighted runs
plot_df$highlight <- !is.na(plot_df[[color_col]]) &
                     plot_df[[color_col]] == highlight_value

p <- ggplot(plot_df, aes(PCA1, PCA2)) +
  geom_point(data = plot_df[!plot_df$highlight, ],
             colour = "grey80", size = 1.5, alpha = 0.5) +
  geom_point(data = plot_df[plot_df$highlight, ],
             aes(colour = .data[[color_col]]), size = 2.5, alpha = 0.9) +
  scale_colour_manual(values = setNames("#D55E00", highlight_value)) +
  labs(x = sprintf("PCA1 (%.1f%%)", var_expl[1]),
       y = sprintf("PCA2 (%.1f%%)", var_expl[2]),
       colour = color_col,
       title = sprintf("PCA of viral family composition (%s)",
                       transfo, highlight_value)) +
  theme_bw()

ggsave("pca_family_date_raw_data.png", p, width = 7, height = 5, dpi = 300)

In [91]:
# Plot, coloured by the chosen metadata column
color_col <- "ASSAY_TYPE"

p <- ggplot(plot_df, aes(PCA1, PCA2, colour = .data[[color_col]])) +
  geom_point(size = 2, alpha = 0.8) +
  labs(x = sprintf("PCA1 (%.1f%%)", var_expl[1]),
       y = sprintf("PCA2 (%.1f%%)", var_expl[2]),
       colour = color_col,
       title = "PCA of viral family composition (%s)") +
  theme_bw()

ggsave("test.png", p) 

Saving 6.67 x 6.67 in image


In [51]:
# keep <- colSums(counts_t > 0) / nrow(counts_t) >= 0.10
# counts_f <- counts_t[rowSums(counts_t[, keep]) > 0, keep]

# Correspondance Analysis
ca <- cca(counts_t)

In [52]:
# Inertia per axis
eig <- ca$CA$eig
var_expl <- eig / sum(eig) * 100

# Weighted orthonormal families scores and column weights
w <- ca$colsum
v <- ca$CA$v

contrib <- w * v[, 1:2]^2 * 100

# Sanity check: each axis sums to 100
colSums(contrib)

sort(contrib[, 1], decreasing = TRUE)[1:15]

CA1 CA2 
100 100

Haloferuviridae   Shortaselviridae      Druskaviridae      Hafunaviridae 
       60.74321966        27.98474016         4.05442712         2.20871343 
      Vertoviridae      Saparoviridae      Madisaviridae      Graaviviridae 
        0.94981085         0.79784747         0.77368738         0.56115245 
      Suolaviridae  Halomagnusviridae    Pleolipoviridae Sphaerolipoviridae 
        0.49730476         0.32946506         0.27018600         0.20468100 
 Autographiviridae      Pyrstoviridae       Kyanoviridae 
        0.12713052         0.12018259         0.09396329

In [53]:
# Inertia per axis
eig <- ca$CA$eig
var_expl <- eig / sum(eig) * 100

# Run and family coordinates (symmetric scaling)
sc_run <- scores(ca, display = "sites", choices = 1:2, scaling = 3)
sc_fam <- scores(ca, display = "species", choices = 1:2, scaling = 3)

runs_df <- data.frame(run = rownames(sc_run), CA1 = sc_run[, 1], CA2 = sc_run[, 2])
fam_df <- data.frame(family = rownames(sc_fam), CA1 = sc_fam[, 1], CA2 = sc_fam[, 2])

# Contribution of each family to axes 1-2, computed by hand from the SVD
P <- counts_f / sum(counts_f)
r <- rowSums(P)
cj <- colSums(P)
S <- (P - r %o% cj) / sqrt(r %o% cj)
sv <- svd(S, nu = 2, nv = 2)
contrib <- sv$v^2 * 100
rownames(contrib) <- colnames(counts_f)

# Top 10 families by total contribution to axes 1 and 2
fam_df$contrib <- rowSums(contrib)[fam_df$family]
top <- fam_df[order(fam_df$contrib, decreasing = TRUE), ][1:10, ]

# Join metadata and flag highlighted runs
runs_df <- merge(runs_df, meta, by = "run", all.x = TRUE)
runs_df$highlight <- !is.na(runs_df[[color_col]]) &
                     runs_df[[color_col]] == highlight_value

p <- ggplot() +
  geom_hline(yintercept = 0, colour = "grey70", linewidth = 0.3) +
  geom_vline(xintercept = 0, colour = "grey70", linewidth = 0.3) +
  geom_point(data = runs_df[!runs_df$highlight, ], aes(CA1, CA2),
             colour = "grey80", size = 1.5, alpha = 0.5) +
  geom_point(data = runs_df[runs_df$highlight, ], aes(CA1, CA2, colour = .data[[color_col]]),
             size = 2.5, alpha = 0.9) +
  geom_point(data = top, aes(CA1, CA2), shape = 17, size = 2.5) +
  geom_text(data = top, aes(CA1, CA2, label = family),
            size = 3, vjust = -0.8, check_overlap = TRUE) +
  scale_colour_manual(values = setNames("#D55E00", highlight_value)) +
  coord_fixed() +
  labs(x = sprintf("CA1 (%.1f%%)", var_expl[1]),
       y = sprintf("CA2 (%.1f%%)", var_expl[2]),
       colour = color_col,
       title = sprintf("CA of viral family composition, highlighted: %s", highlight_value)) +
  theme_bw()

ggsave("ca_family_date.png", p, width = 7, height = 5, dpi = 300)

Warning message:
“No shared levels found between `names(values)` of the manual scale and the
data's colour values.”
